# TP Final — Du dataset au modèle déployé
**Formation Machine Learning & Data Science — Jour 3**

PAR: KAMGA ZAINAB


## PARTIE A — Chargement & choix des variables

### Étape 1 — Charger le fichier Excel et le convertir en CSV

In [1]:
import pandas as pd, numpy as np
import warnings; warnings.filterwarnings('ignore')

from pathlib import Path

cwd = Path.cwd()
if (cwd / 'data').exists():
    ROOT = cwd
elif (cwd.parent / 'data').exists():
    ROOT = cwd.parent
else:
    raise FileNotFoundError("Dossier data/ introuvable. Vérifie l'emplacement du notebook.")

DATA_DIR = ROOT / 'data'
MODELS_DIR = ROOT / 'models'
MODELS_DIR.mkdir(parents=True, exist_ok=True)
print('Racine du projet détectée :', ROOT)

df = pd.read_excel(DATA_DIR / 'dataset_assurance_ML.xlsx')
df.to_csv(DATA_DIR / 'dataset_assurance_ML.csv', index=False, encoding='utf-8-sig')

df = pd.read_csv(DATA_DIR / 'dataset_assurance_ML.csv', encoding='utf-8-sig')
print(df.shape)
print('Valeurs manquantes :', df.isnull().sum().sum())
print('Doublons :', df.duplicated().sum())

Racine du projet détectée : c:\Users\kamga\Desktop\scoring_resiliation
(500, 27)
Valeurs manquantes : 0
Doublons : 0


**Constat :** 500 lignes × 27 colonnes, 0 valeur manquante, 0 doublon — c'est bien le fichier nettoyé au Jour 2.

### Étape 2 — La variable cible

In [2]:
TARGET = 'Résiliation'
print(df[TARGET].value_counts())
print(df[TARGET].value_counts(normalize=True).round(2))

Résiliation
0    450
1     50
Name: count, dtype: int64
Résiliation
0    0.9
1    0.1
Name: proportion, dtype: float64


**Question :** le problème est-il équilibré ? Quelle conséquence pour l'entraînement ?

**Réponse :** non, il est très déséquilibré : 450 clients restent (90 %) contre seulement 50 qui résilient (10 %). Un modèle entraîné naïvement aurait tendance à toujours prédire « reste », puisque c'est déjà juste avec 90 % de précision sans rien apprendre. Deux conséquences pratiques : il faut utiliser `class_weight='balanced'` pour forcer le modèle à accorder plus de poids aux rares cas de résiliation, et il faut évaluer avec le ROC-AUC (ou le F1) plutôt qu'avec l'accuracy, qui serait trompeuse ici.

### Étape 3 — Détecter une fuite de données (data leakage)

In [3]:
print(pd.crosstab(df['Statut Contrat'], df[TARGET]))

Résiliation       0   1
Statut Contrat         
Actif           450   0
Résilié           0  36
Suspendu          0  14


**Question :** que remarquez-vous ? Peut-on utiliser `Statut Contrat` pour prédire la résiliation ?

**Réponse :** non, absolument pas. Le tableau croisé le montre sans ambiguïté : `Actif` correspond à 450 clients tous à `Résiliation=0`, `Résilié` à 36 clients tous à `Résiliation=1`, `Suspendu` à 14 clients tous à `Résiliation=1`. Cette colonne prédit la cible à 100 %, ce qui est logique — c'est littéralement le même événement raconté deux fois. Le problème n'est pas la performance (un modèle qui l'utiliserait serait « parfait » en test), mais l'utilité réelle : au moment où on veut anticiper la résiliation d'un client encore actif, on ne connaît pas encore ce statut futur. L'inclure donnerait un modèle inutilisable en production.

### Étape 4 — Choisir les variables numériques et catégorielles

In [4]:
num_cols = ['Âge', 'Salaire Annuel (€)', 'Prime Annuelle (€)', 'Ancienneté (mois)',
            'Coeff. Bonus-Malus', 'Nb Sinistres (3 ans)',
            'Montant Sinistres (€)', 'Score Risque (0-100)']
cat_cols = ['Type Contrat', 'Catégorie Prof.', 'Usage Véhicule', 'Dernier Sinistre']

X = df[num_cols + cat_cols]
y = df[TARGET]
print(X.shape, y.shape)

(500, 12) (500,)


### Étape 5 — Vérifier le lien avec la cible

In [5]:
print(X[num_cols].corrwith(y).round(3).sort_values(ascending=False))
print()
print(df.groupby('Dernier Sinistre')[TARGET].mean().round(2).sort_values())

Nb Sinistres (3 ans)     0.455
Score Risque (0-100)     0.441
Coeff. Bonus-Malus       0.412
Montant Sinistres (€)    0.284
Prime Annuelle (€)       0.041
Âge                     -0.006
Salaire Annuel (€)      -0.012
Ancienneté (mois)       -0.055
dtype: float64

Dernier Sinistre
Aucun                    0.03
Incendie                 0.16
Catastrophe naturelle    0.18
Accident                 0.26
Bris de glace            0.30
Vol                      0.30
Dégât des eaux           0.36
Name: Résiliation, dtype: float64


**Question :** quelles sont les 3 variables numériques les plus liées à la résiliation ?

**Réponse :** sur ces données, ce sont `Nb Sinistres (3 ans)` (≈ 0,455), `Score Risque (0-100)` (≈ 0,441) et `Coeff. Bonus-Malus` (≈ 0,412) qui arrivent en tête — toutes trois racontent la même histoire (la sinistralité du client), ce qui est cohérent puisque le score de risque et le bonus-malus sont eux-mêmes construits à partir de l'historique de sinistres. Côté catégoriel, `Dernier Sinistre` est très parlant : les clients avec `Aucun` sinistre résilient à 3 %, contre 30-36 % pour ceux dont le dernier sinistre est `Dégât des eaux` ou `Vol` — un écart net qui confirme que le passif du client pèse plus que son profil socio-démographique (âge et salaire sont quasiment non corrélés, autour de 0).

## PARTIE B — Pipeline, entraînement & évaluation

### Étape 6 — Séparer train / test

In [6]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

print(X_train.shape, X_test.shape)
print(y_train.mean().round(2), y_test.mean().round(2))

(400, 12) (100, 12)
0.1 0.1


**Constat :** le split stratifié conserve bien ~10 % de résiliation dans les deux jeux (voir sortie ci-dessus).

### Étape 7 — Le prétraitement en un seul objet : ColumnTransformer

In [7]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

preprocessor = ColumnTransformer([
    ('num', StandardScaler(), num_cols),
    ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols),
])

### Étape 8 — Deux candidats dans un Pipeline

In [8]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

candidats = {
    'Régression Logistique': LogisticRegression(
        max_iter=1000, class_weight='balanced', random_state=42),
    'Random Forest': RandomForestClassifier(
        n_estimators=300, max_depth=4, min_samples_leaf=10,
        class_weight='balanced', random_state=42),
}

pipelines = {nom: Pipeline([('prep', preprocessor), ('model', algo)])
             for nom, algo in candidats.items()}
print(list(pipelines.keys()))

['Régression Logistique', 'Random Forest']


### Étape 9 — Comparer par validation croisée

In [9]:
from sklearn.model_selection import cross_val_score

resultats_cv = {}
for nom, pipe in pipelines.items():
    scores = cross_val_score(pipe, X_train, y_train, cv=5, scoring='roc_auc')
    resultats_cv[nom] = (scores.mean(), scores.std())
    print(f'{nom:22s} AUC = {scores.mean():.3f} ± {scores.std():.3f}')

Régression Logistique  AUC = 0.745 ± 0.113
Random Forest          AUC = 0.806 ± 0.096


**Question :** lequel retenez-vous ? L'écart est-il significatif ?

**Réponse :** les deux scores se chevauchent largement une fois qu'on prend en compte l'écart-type — l'écart entre les deux modèles n'est pas significatif sur seulement 400 lignes de train. Je retiens quand même le **Random Forest** : à performance équivalente, il a l'avantage de fournir `feature_importances_`, directement utile pour l'onglet d'explication de l'interface Streamlit (partie D), ce qu'une régression logistique n'offre pas aussi naturellement.

### Étape 10 — Entraîner le modèle retenu et évaluer sur le test

In [10]:
from sklearn.metrics import (accuracy_score, f1_score, roc_auc_score,
                             confusion_matrix, classification_report)

pipeline = pipelines['Random Forest']
pipeline.fit(X_train, y_train)

y_pred = pipeline.predict(X_test)
y_proba = pipeline.predict_proba(X_test)[:, 1]

print('Accuracy :', round(accuracy_score(y_test, y_pred), 3))
print('F1 :', round(f1_score(y_test, y_pred), 3))
print('ROC-AUC :', round(roc_auc_score(y_test, y_proba), 3))
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred, target_names=['Reste', 'Résilie']))

Accuracy : 0.82
F1 : 0.438
ROC-AUC : 0.861
[[75 15]
 [ 3  7]]
              precision    recall  f1-score   support

       Reste       0.96      0.83      0.89        90
     Résilie       0.32      0.70      0.44        10

    accuracy                           0.82       100
   macro avg       0.64      0.77      0.67       100
weighted avg       0.90      0.82      0.85       100



### Étape 11 — Lire la matrice de confusion comme un métier

**Q1. Un modèle qui prédirait toujours « reste » aurait 90 % d'accuracy. Est-il meilleur que le mien ?**

Non — cette accuracy de 90 % est un piège classique du déséquilibre de classes : ce modèle-la ne détecterait *aucun* client à risque (recall = 0 % sur la classe résiliation), il serait donc inutile pour le service Fidélisation dont le but est justement de repérer ces clients-là. Mon modèle a une accuracy plus faible sur le papier, mais c'est le prix à payer pour détecter une partie des vraies résiliations — voir la matrice de confusion ci-dessus pour le détail exact obtenu sur ce jeu de test.

**Q2. Pour le service Fidélisation, quelle erreur coûte le plus cher : rater un client qui va partir (faux négatif) ou appeler un client qui serait resté (faux positif) ?**

Le faux négatif coûte plus cher. Un faux positif, c'est un appel « pour rien » à un client fidèle — un peu de temps conseiller perdu, tout au plus une gêne mineure. Un faux négatif, c'est un client qui part sans qu'on ait rien tenté, avec le coût complet d'acquisition d'un nouveau client pour le remplacer, généralement bien plus élevé qu'un appel de rétention.

**Q3. Faut-il donc plutôt baisser ou monter le seuil de 0,5 ?**

Baisser le seuil. Puisque rater un résiliataire coûte plus cher qu'un faux appel, il vaut mieux que le modèle déclenche l'alerte plus facilement — quitte à multiplier les faux positifs — pour maximiser le nombre de vrais départs interceptés (le recall), même si cela dégrade la précision. C'est cette logique qui justifie le curseur de seuil ajustable proposé en défi bonus de l'interface : le conseiller peut alors choisir lui-même ce compromis plutôt que de subir un seuil fixe à 0,5.

## PARTIE C — Sauvegarde & interrogation du modèle

### Étape 12 — Sauvegarder le pipeline complet

In [11]:
import joblib, os

joblib.dump(pipeline, MODELS_DIR / 'pipeline_resiliation.pkl')
taille_ko = os.path.getsize(MODELS_DIR / 'pipeline_resiliation.pkl') / 1024
print(f'{taille_ko:.1f} Ko')

486.8 Ko


### Étape 13 — Sauvegarder les métadonnées pour l'interface

In [12]:
import json

meta = {
    'modele': 'Random Forest',
    'auc_test': round(float(roc_auc_score(y_test, y_proba)), 3),
    'num_cols': num_cols,
    'cat_cols': cat_cols,
    'num_ranges': {c: {'min': float(X[c].min()), 'max': float(X[c].max()),
                        'median': float(X[c].median())} for c in num_cols},
    'cat_values': {c: sorted(X[c].unique().tolist()) for c in cat_cols},
}

with open(MODELS_DIR / 'metadata.json', 'w', encoding='utf-8') as f:
    json.dump(meta, f, ensure_ascii=False, indent=2)

print(json.dumps(meta, ensure_ascii=False, indent=2)[:600], '...')

{
  "modele": "Random Forest",
  "auc_test": 0.861,
  "num_cols": [
    "Âge",
    "Salaire Annuel (€)",
    "Prime Annuelle (€)",
    "Ancienneté (mois)",
    "Coeff. Bonus-Malus",
    "Nb Sinistres (3 ans)",
    "Montant Sinistres (€)",
    "Score Risque (0-100)"
  ],
  "cat_cols": [
    "Type Contrat",
    "Catégorie Prof.",
    "Usage Véhicule",
    "Dernier Sinistre"
  ],
  "num_ranges": {
    "Âge": {
      "min": 18.0,
      "max": 79.0,
      "median": 50.0
    },
    "Salaire Annuel (€)": {
      "min": 14000.0,
      "max": 90433.0,
      "median": 36117.0
    },
    "Prime Annuelle  ...


### Étape 14 — Interroger le modèle sur un nouveau client

In [13]:
# Nouvelle cellule, comme si on repartait de zéro
modele = joblib.load(MODELS_DIR / 'pipeline_resiliation.pkl')

client_risque = pd.DataFrame([{
    'Âge': 34, 'Salaire Annuel (€)': 28000, 'Prime Annuelle (€)': 950,
    'Ancienneté (mois)': 6, 'Coeff. Bonus-Malus': 1.25, 'Nb Sinistres (3 ans)': 3,
    'Montant Sinistres (€)': 4200, 'Score Risque (0-100)': 72,
    'Type Contrat': 'Bronze', 'Catégorie Prof.': 'Entrepreneur',
    'Usage Véhicule': 'Professionnel', 'Dernier Sinistre': 'Vol',
}])
print('Client à risque — Classe :', modele.predict(client_risque))
print('Client à risque — Proba  :', round(modele.predict_proba(client_risque)[0, 1], 3))

client_fidele = pd.DataFrame([{
    'Âge': 55, 'Salaire Annuel (€)': 42000, 'Prime Annuelle (€)': 700,
    'Ancienneté (mois)': 200, 'Coeff. Bonus-Malus': 0.6, 'Nb Sinistres (3 ans)': 0,
    'Montant Sinistres (€)': 0, 'Score Risque (0-100)': 10,
    'Type Contrat': 'Gold', 'Catégorie Prof.': 'Cadre',
    'Usage Véhicule': 'Privé', 'Dernier Sinistre': 'Aucun',
}])
print('Client fidèle — Classe :', modele.predict(client_fidele))
print('Client fidèle — Proba  :', round(modele.predict_proba(client_fidele)[0, 1], 3))

Client à risque — Classe : [1]
Client à risque — Proba  : 0.828
Client fidèle — Classe : [0]
Client fidèle — Proba  : 0.223


**Constat :** les deux profils donnent des probabilités nettement différentes (voir sortie ci-dessus) et dans le sens attendu — le profil jeune, peu ancien, avec plusieurs sinistres et un score de risque élevé ressort à un risque bien plus élevé que le profil ancien, sans sinistre. Les deux DataFrames sont construits avec des données **brutes** (texte, euros) : c'est le pipeline lui-même, via le `ColumnTransformer` sauvegardé dedans, qui s'occupe de tout le prétraitement — aucune transformation manuelle n'est nécessaire côté appelant.

### Étape 15 — Provoquer l'erreur classique, puis passer en script

In [14]:
try:
    modele.predict(client_risque.drop(columns=['Score Risque (0-100)']))
except Exception as e:
    print('ERREUR :', e)

ERREUR : columns are missing: {'Score Risque (0-100)'}


**Question :** quelle règle en tirez-vous pour l'interface ?

**Réponse :** le pipeline exige EXACTEMENT les colonnes vues à l'entraînement, avec les mêmes noms — voir l'erreur ci-dessus. Ça veut dire que l'interface Streamlit doit impérativement collecter les 12 variables de `num_cols + cat_cols`, ni plus ni moins, avec l'orthographe exacte (accents, espaces, parenthèses compris) — une seule colonne manquante ou mal nommée fait planter la prédiction. C'est justement pour éviter ce genre d'erreur de recopie que l'app lira les noms de colonnes depuis `metadata.json` plutôt que de les retaper en dur.

Le code des parties A, B, C ci-dessus est regroupé (sans les cellules exploratoires) dans `train_model.py`, à la racine du projet — il recrée `pipeline_resiliation.pkl` et `metadata.json` en une seule commande : `python train_model.py`.

## Étape 23 — Tester comme un conseiller (interface Streamlit)

Trois profils testés directement dans l'application (`streamlit run app.py`), en relevant la probabilité affichée après clic sur **Prédire** :

| Profil | Probabilité | Message |
|---|---|---|
| (a) Profil médian (curseurs par défaut) | **28 %** | ✅ Client fidèle — risque faible |
| (b) Jeune conducteur Bronze, 2 sinistres, Vol | **57 %** | ⚠️ Client À RISQUE — action de rétention conseillée |
| (c) Client Gold ancien, sans sinistre | **11 %** | ✅ Client fidèle — risque faible |

**Question : les résultats sont-ils cohérents avec les corrélations de l'étape 5 ?**

**Réponse :** oui, complètement. À l'étape 5, les variables les plus corrélées à la résiliation étaient `Nb Sinistres (3 ans)` (0,455), `Score Risque (0-100)` (0,441) et `Coeff. Bonus-Malus` (0,412), et `Dernier Sinistre = Vol` ressortait à un taux de résiliation de 30 % contre 3 % pour `Aucun`. Le profil (b) cumule justement de mauvaises valeurs sur ces quatre variables (2 sinistres, score élevé, bonus-malus dégradé, dernier sinistre = Vol) et obtient la probabilité la plus haute des trois (57 %). Le profil (c), à l'opposé — 0 sinistre, `Dernier Sinistre = Aucun` — obtient la plus basse (11 %). Le profil médian (a) se situe logiquement entre les deux (28 %). L'ordre b > a > c est exactement celui que les corrélations de l'étape 5 laissaient prévoir.

**Question : que se passe-t-il si vous changez uniquement la Ville dans votre esprit ?**

**Réponse :** rien ne changerait dans la prédiction, parce que `Ville` ne fait pas partie des 12 variables retenues à l'étape 4 (`num_cols` + `cat_cols`). Elle n'apparaît donc même pas dans l'interface — pas de curseur, pas de menu pour elle. C'est un choix assumé dès la sélection des variables : le TP a volontairement limité l'interface à 12 champs, en ne gardant que les variables les plus prédictives (sinistralité, contrat) plutôt que des variables démographiques comme la ville, qui n'étaient pas ressorties comme discriminantes au Jour 2.